# Open-source climate data — IMD gridded exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anantrajj7-sketch/lecture-deck/blob/main/imd_gridded_clip_exercise.ipynb)

Downloading IMD gridded rainfall data with `imdlib`, opening it as an `xarray` grid, and clipping it to a study area with a 33 km buffer around a shapefile.

Run each cell top to bottom. Cells you need to edit for your own study area are marked **EDIT ME**.


## 1. Install the libraries

Nothing is pre-installed in a fresh Colab runtime — this cell pulls in everything we need. Only needs to run once per session.


In [ ]:
!pip install -q imdlib geopandas shapely


## 2. Download the IMD gridded data

`imd.get_data` fetches the raw `.GRD` files from IMD Pune, one per year, and caches them locally. **EDIT ME**: change the variable (`'rain'`, `'tmin'`, or `'tmax'`) and the year range for your own use case.


In [ ]:
import imdlib as imd

variable = 'rain'      # 'rain', 'tmin', or 'tmax'
start_year = 2020
end_year = 2023

imd.get_data(variable, start_year, end_year, fn_format='yearwise')
data = imd.open_data(variable, start_year, end_year, 'yearwise')
ds = data.get_xarray()
ds

# IMD's fill value for missing/out-of-domain cells is -999, not NaN.
# Verified against a real download: get_xarray() leaves it raw (millions of
# -999 cells, zero NaNs) -- mask it ourselves before it can pollute an average.
ds[variable] = ds[variable].where(ds[variable] != -999)
ds


Before moving on: IMD's grid uses `-999` to mark cells with no data (mostly ocean, outside India's landmass) — not `NaN`. `get_xarray()` does **not** convert this for you, so the cell above masks it explicitly. Skipping this step means `-999` can silently wreck a `.mean()` later.


## 3. Recall: pulling a single point

This is the part you've already done before — picking one lat/lon out of the grid. It's a useful sanity check that the download worked before we move on to clipping a whole region.


In [ ]:
# EDIT ME: your point of interest
point_lat, point_lon = 28.6, 77.2   # Delhi, as an example

point_series = ds[variable].sel(lat=point_lat, lon=point_lon, method='nearest')
point_series.to_dataframe().reset_index().head()


## 4. Load your shapefile

Upload a **zipped** shapefile — a `.zip` containing the `.shp`, `.shx`, `.dbf`, and (ideally) `.prj` files together. Zipping avoids having to upload four separate files one at a time.

No shapefile handy yet? Skip to the cell below marked "no shapefile yet" to try the exercise with a small example polygon instead.


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # choose your shapefile .zip
zip_name = next(iter(uploaded))

extract_dir = 'shapefile'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

shp_files = [f for f in os.listdir(extract_dir) if f.endswith('.shp')]
shp_path = os.path.join(extract_dir, shp_files[0])
print('Using shapefile:', shp_path)


In [ ]:
import geopandas as gpd

gdf = gpd.read_file(shp_path).to_crs(epsg=4326)
gdf.plot()


### No shapefile yet?

Run this instead of the two cells above to try the whole exercise with a small example boundary (a box around Delhi) — replace it with your real shapefile once you have one.


In [ ]:
# import geopandas as gpd
# from shapely.geometry import box
#
# gdf = gpd.GeoDataFrame(geometry=[box(76.9, 28.4, 77.5, 28.9)], crs='EPSG:4326')
# gdf.plot()


## 5. Buffer the shapefile by 33 km

Buffering in degrees is wrong — a degree of longitude isn't a fixed distance, so the buffer would be a different real-world size depending on latitude. `estimate_utm_crs()` picks the right metric (meters-based) projection automatically, so we can buffer in meters and reproject back.


In [ ]:
BUFFER_METERS = 33_000   # EDIT ME if you need a different buffer distance

utm_crs = gdf.estimate_utm_crs()
buffered_metric = gdf.to_crs(utm_crs).buffer(BUFFER_METERS)
boundary = gpd.GeoSeries(buffered_metric, crs=utm_crs).to_crs(epsg=4326).union_all()
boundary


## 6. Turn every grid cell into a point

To know which grid cells fall inside the buffered boundary, we need every cell as a point we can test — not just the four corners of the grid.


In [ ]:
import numpy as np
from shapely.geometry import Point

lat_vals = ds.lat.values
lon_vals = ds.lon.values
lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

grid_points = gpd.GeoDataFrame(
    geometry=[Point(x, y) for x, y in zip(lon2d.ravel(), lat2d.ravel())],
    crs='EPSG:4326',
)
len(grid_points)


## 7. Keep only the points inside the buffer

`.within()` tests every point against the boundary polygon in one vectorized call, giving a True/False mask the same shape as the grid.


In [ ]:
import xarray as xr

inside_flat = grid_points.within(boundary).values
inside_2d = inside_flat.reshape(lat2d.shape)

mask = xr.DataArray(inside_2d, dims=['lat', 'lon'], coords={'lat': lat_vals, 'lon': lon_vals})
print('Grid points kept:', int(inside_2d.sum()), 'of', inside_2d.size)


## 8. Apply the mask and extract the clipped points

`.where()` keeps the grid's shape but sets everything outside the mask to `NaN`; dropping those rows leaves only the points near your study area.


In [ ]:
clipped = ds[variable].where(mask)
points_df = clipped.to_dataframe().reset_index().dropna(subset=[variable])
points_df.head()


## 9. Visual check

Plot the shapefile, the 33 km buffer, and the surviving grid points together — a quick way to confirm the clip did what you expect before trusting the numbers.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
gpd.GeoSeries([boundary], crs='EPSG:4326').boundary.plot(ax=ax, color='tab:orange', linewidth=2, label='33 km buffer')
gdf.boundary.plot(ax=ax, color='black', linewidth=1, label='shapefile')
ax.scatter(points_df['lon'], points_df['lat'], s=8, color='tab:blue', label='kept grid points')
ax.legend()
ax.set_title('Grid points retained after clipping')
plt.show()


## 10. Save the result

`points_df` has one row per (time, lat, lon) that survived the clip. `-999` fill values were already dropped along with the other NaNs in step 8, so this is safe to average or export as-is.


In [ ]:
OUTPUT_PATH = 'clipped_points.csv'   # EDIT ME — point this at Drive if you want it to survive the session

points_df.to_csv(OUTPUT_PATH, index=False)
print('Saved', len(points_df), 'rows to', OUTPUT_PATH)


## Recap

- Point vs. gridded data — you pulled both a single point (step 3) and clipped a full region (steps 4–8) from the same downloaded grid.
- The grid was never "just your basin" — imdlib always downloads the whole country; clipping is something *you* do afterward.
- Buffering must happen in a metric CRS, not directly in degrees.
- `-999` is IMD's fill value, not `NaN` — always mask it before averaging.
